In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [8]:
#hvem er nico

df = pd.read_csv("Cleaned_again.csv")
X = df.iloc[:,1:]
y = df["ClaimNb"]
y

0         0
1         0
2         0
3         0
4         0
         ..
541411    0
541412    0
541413    0
541414    0
541415    0
Name: ClaimNb, Length: 541416, dtype: int64

In [47]:
#fan_int = features, fan_out = output
#Kaiming init/HE init
#Used to initialize weights for one hidden layer.
def he_init(fan_in, fan_out):
    std = np.sqrt(2 / fan_in)
    W = np.random.normal(0, std, size=(fan_out, fan_in))
    return W

#Activiation fucntion
def ReLu(arr):
    return np.maximum(0,arr)

#Cost function
def MSE(y_true,y_pred):
    return ((y_true-y_pred)**2).mean()

def forward_pass(X,W,b):
    #Combination
    z= X.dot(W.T)+b
    a = ReLu(z)
    return z,a


In [32]:
he_init(6,4)

array([[ 0.89321597, -0.57619931,  0.55207833, -0.51749603, -0.64396041,
        -0.11774584]])

***What this really needs is a way of remembering which weights and biases, and such corresponds to which layer***

**A ToDo, which I will do later/friday or sunday, is add this for the implementation and then try backpropagation**

$\frac{dL}{dZ2}=\frac{2}{m}*(A2-y_{true})$ <- **MSE Gradient**

In [1]:
""" 
Feed-forward
Get p number of features as input layer neurons. Decicde the number of hidden layers and neurons we have in each.
For our we have right now 1 hidden layers with p-2 neurons. For that we will get a linear combination
of weights and inputs for each neuron in the hidden layer. So each input neuron outputs to 4 neurons.

We use the ReLu function to return, a, which is the new values for input. We randomly init weights for that layer using
HE init. With all the a's we compute a linear combination for the output.

Did the math calculations, the logic and with help of LLM to implement it using python.
"""

#1 hidden layer
class neural:
    def __init__(self, input_size, hidden_size):
        self.input_size = input_size
        self.hidden_size = hidden_size
        output_size = 1
        self.weights = {}
        self.bias = {}
        #Creates weight matrix, using HE init. This uses a normal distribution
        #Bias is set to 0 at the start.

        for i in range(len(hidden_size)):
            if self.hidden_size[i] == self.hidden_size[0]:
                self.weights["W" + 1]= np.random.normal(0, np.sqrt(2/input_size), (input_size, hidden_size[i]))
                self.bias["bias" + 1] = np.zeros((1, hidden_size[i]))
            self.weights["W" + i+1] = np.random.normal(0, np.sqrt(2/hidden_size[i-1]), (hidden_size[i-1], hidden_size[i]))
            self.bias["bias" + i+1] = np.zeros((1, hidden_size[i]))

            self.W2 = np.random.normal(0, np.sqrt(2/hidden_size), (hidden_size, output_size))
            self.b2 = np.zeros((1, output_size))
        #A cache to keep track of the parameters
        self.cache = {}
        self.grad = {}
        
    
 
    def printW(self):
        print("Weight 1: ", self.W1, "Weight 2: ", self.W2)
    #Cost function
    def MSE(self, y_true,y_pred):
        cost = ((y_true-y_pred)**2).mean()
        return cost
    #Activation function
    def ReLu(self, Z):
        return np.maximum(0,Z)
    
    def grad_ReLu(self, Z):
        return (Z>0).astype(float)
    
    def forward_pass(self, X):
        #First A as X/features
        A = X
        #Save A0 as X, for backpropagation: dWi = A^Ti-1 dot dZi
        self.cache["A0"] = X

        L = len()
        for i in range(1,L+1):
            W = self.weights[f"W{i}"]
            b = self.bias[f"b{i}"]
            Z = A.dot(W) + b
            self.cache[f"Z{i}"] = Z

            if i != L:
                A = self.ReLu(Z)
            else:
                A = Z
            self.cache[f"A{i}"]
        return A

    def backward(self, y_true, learning_rate = 0.001,lambda_reg = 0.001):
        X = self.cache["X"]
        Z1 = self.cache["Z1"]
        A1 = self.cache["A1"]
        Z2 = self.cache["Z2"]
        A2 = self.cache["A2"]
        y_true = y_true.reshape(-1, 1)
        m = y_true.shape[0]

        L = len(self.weights)
        A = self.cache[f"A{L}"]
        dA = (2/m)*(A-y_true)
        for i in reversed(range(1,L+1)):
            Z = self.cache[f"Z{i}"]
            A_prev = self.cache[f"A{i-1}"]
            if i == L:
                dZ = dA
            else:
                dZ = dA * self.grad_ReLu(Z)
            dW = A_prev.T.dot(dZ)
            db = np.sum(dZ, axis=0, keepdims=True) / m
            dA = dZ.dot(self.weights[f"W{i}"].T)
            self.grad[f"dW{i}"] = dW
            self.grad[f"db{i}"] = db
            
        for i in range(1,L+1):
            self.weights[f"W{i}"] -= learning_rate * self.grad[f"dW{i}"]
            self.bias[f"b{i}"] -= learning_rate * self.grad[f"db{i}"]


        #Output gradient/MSE, with respect to Z2
        #A2 would be the predicted value
        dZ2 = (2/m)*(A2-y_true)

        dW2 = A1.T.dot(dZ2) / m
        db2 = np.sum(dZ2, axis=0, keepdims=True) / m


        dA1 = dZ2.dot(self.W2.T)
        dZ1 = dA1 * self.grad_ReLu(Z1)

        dW1 = X.T.dot(dZ1) / m
        db1 = np.sum(dZ1, axis=0, keepdims=True) / m

        dW2 += (lambda_reg / m) * self.W2 
        dW1 += (lambda_reg / m) * self.W1

        # update weights and biases
        self.W1 -= learning_rate * dW1
        self.b1 -= learning_rate * db1
        self.W2 -= learning_rate * dW2
        self.b2 -= learning_rate * db2
    def train(self, X, y, epochs = 1000, learning_rate = 0.01):
        X = np.array(X)
        y = np.array(y)
        m = X.shape[0]  # number of examples
        y = y.reshape(-1, 1)    

        for epoch in range(epochs):
            #Using the forward pass function
            A2 = self.forward_pass(X)

            #Monitoring the loss
            loss = np.mean((A2 - y)**2)

            #Doing the backward propagation
            self.backward(y, learning_rate)

            #Right now printing for every 100 epoch, the loss.
            if epoch % 100 == 0:
                print(f"Epoch {epoch}, Loss: {loss:.6f}")
    def train_batch(self, X,y,mini_batch_size ,epochs=1000, learning_rate = 0.01):
    # Convert to numpy arrays to avoid indexing errors
        X = np.array(X)
        y = np.array(y)
        
        #Ensure y is the correct shape (N, 1)
        if y.ndim == 1:
            y = y.reshape(-1, 1)
            
        m = X.shape[0]  # Total number of examples

        #Ensure inputs are same length
        if X.shape[0] != y.shape[0]:
            raise ValueError(f"Shape Mismatch: X has {X.shape[0]} rows, but y has {y.shape[0]} rows.")

        for epoch in range(epochs):
            # Shuffle data at the start of each epoch
            permutation = np.random.permutation(m)
            X_shuffled = X[permutation, :]
            Y_shuffled = y[permutation]

            # 3. The Clean Loop: Step through data by batch size
            # This handles both full batches AND the leftover remainder automatically.
            for start in range(0, m, mini_batch_size):
                end = start + mini_batch_size
                
                
                # Python slicing [start:end] handles "end" being too large automatically
                X_batch = X_shuffled[start:end, :]
                Y_batch = Y_shuffled[start:end]

                self.forward_pass(X_batch)
                
                # Backward pass
                self.backward(Y_batch, learning_rate)

            # Optional: Print loss occasionally to verify training
            if epoch % 100 == 0:
                # Calculate loss on the LAST batch of the epoch
                # (Or you could compute it on the whole dataset, but that's slower)
                A2 = self.cache["A2"]
                loss = np.mean((A2 - Y_batch)**2)
                print(f"Epoch {epoch}, Batch Loss: {loss:.6f}")
    def predict(self, X):
            # 1. Ensure input is a numpy array
            X = np.array(X)
            
            # 2. Layer 1 (Hidden)
            # Calculate Z1 (Linear step)
            Z1 = X.dot(self.W1) + self.b1
            # Calculate A1 (Activation step - ReLU)
            A1 = np.maximum(0, Z1) 
            
            # 3. Layer 2 (Output)
            # Calculate Z2 (Linear step)
            Z2 = A1.dot(self.W2) + self.b2
            
            # Since the output activation is linear (for MSE), 
            # the prediction is just Z2.
            return Z2




In [5]:
nn = neural(6,4)
nn.printW()

Weight 1:  [[ 0.76683244 -0.30349942  0.62889675 -1.3681562 ]
 [ 0.36253797  0.62330238 -0.04183233  0.36620706]
 [ 0.17391547  0.57644158 -0.32189465  0.02304621]
 [ 0.40780979  0.47623351  0.63055204  0.20429255]
 [ 0.59029216  0.58771718 -0.32011877  0.23140246]
 [ 0.09460339 -0.52418452 -0.69951451 -0.0259849 ]] Weight 2:  [[-0.85444204]
 [-0.14280105]
 [-0.58730764]
 [ 0.3097895 ]]


In [7]:
if __name__ == "__main__":
    # Create dummy data
    X = np.random.randn(100000, 10)  # 100 samples, 10 features
    y = np.random.randn(100000)      # 100 targets (Rank 1 array)

    # Initialize
    nn = neural(input_size=10, hidden_size=5)
    
    # Train (Should run without crashing now)
    nn.train(X, y, epochs=500, learning_rate=0.01)

Epoch 0, Loss: 3.386515
Epoch 100, Loss: 3.386310
Epoch 200, Loss: 3.386104
Epoch 300, Loss: 3.385899
Epoch 400, Loss: 3.385694


In [29]:
from sklearn.preprocessing import StandardScaler
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [34]:
new_n = neural(6,4)
new_n.train_batch(X_train_scaled,y_train, mini_batch_size=100,epochs=5000)

Epoch 0, Batch Loss: 0.226336
Epoch 100, Batch Loss: 0.030492
Epoch 200, Batch Loss: 0.055679
Epoch 300, Batch Loss: 0.058489
Epoch 400, Batch Loss: 0.003120
Epoch 500, Batch Loss: 0.030900
Epoch 600, Batch Loss: 0.031220
Epoch 700, Batch Loss: 0.031442
Epoch 800, Batch Loss: 0.003370
Epoch 900, Batch Loss: 0.059477
Epoch 1000, Batch Loss: 0.031793
Epoch 1100, Batch Loss: 0.029863
Epoch 1200, Batch Loss: 0.030110
Epoch 1300, Batch Loss: 0.030876
Epoch 1400, Batch Loss: 0.058360
Epoch 1500, Batch Loss: 0.059980
Epoch 1600, Batch Loss: 0.056642
Epoch 1700, Batch Loss: 0.058209
Epoch 1800, Batch Loss: 0.003308
Epoch 1900, Batch Loss: 0.002301
Epoch 2000, Batch Loss: 0.002823
Epoch 2100, Batch Loss: 0.004143
Epoch 2200, Batch Loss: 0.053493
Epoch 2300, Batch Loss: 0.031887
Epoch 2400, Batch Loss: 0.084240
Epoch 2500, Batch Loss: 0.002907
Epoch 2600, Batch Loss: 0.031550
Epoch 2700, Batch Loss: 0.082541
Epoch 2800, Batch Loss: 0.084185
Epoch 2900, Batch Loss: 0.059691
Epoch 3000, Batch Loss

In [37]:
y_test = np.array(y_test)
if y_test.ndim == 1:
    y_test = y_test.reshape(-1, 1)
y_test

array([[0],
       [0],
       [1],
       ...,
       [0],
       [0],
       [0]], shape=(108284, 1))

In [38]:
y_pred = new_n.predict(X_test_scaled)
mse = np.mean((y_pred - y_test)**2)


In [39]:
y_mean = np.mean(y_test)

# Sum of Squares Residual (SSR) - Error from your model
SS_residual = np.sum((y_test - y_pred)**2)

# Sum of Squares Total (SST) - Error from the baseline mean
SS_total = np.sum((y_test - y_mean)**2)

# R-squared calculation
r2 = 1 - (SS_residual / SS_total)
r2

np.float64(0.013880220042027491)

In [9]:
oo = [1,2,3,4]
last = oo[-1]
last

4

In [12]:
for i in range(5):
    if i == 2:
        print("hi")
        continue
    print("oi")

oi
oi
hi
oi
oi
